# Assignment 1  

For the strat of the assignment, we need to import pandas and numpy in order to be able to inspect and clean the data. We also need to import the dataset, which is in this case assigned to **transactions**.

In [1]:
import pandas as pd
import numpy as np
transactions = pd.read_csv("../data/transaction_data.csv")

## Question 1  - Data Frame formatting

Before starting any work on the data, we need to inspect it so that we know how the data is structured.

In [2]:
transactions.info()
transactions.sample(5)

<class 'pandas.DataFrame'>
RangeIndex: 760460 entries, 0 to 760459
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   sku_numb3r          757837 non-null  str    
 1   inventory_type      747418 non-null  str    
 2   stocking_type       745287 non-null  str    
 3   lead_time           760460 non-null  int64  
 4   unit_price          760460 non-null  float64
 5   Manufacturing Site  612982 non-null  str    
 6   division Code       707229 non-null  str    
 7   transaction_date    760460 non-null  str    
 8   order_quantity      760460 non-null  int64  
dtypes: float64(1), int64(2), str(6)
memory usage: 52.2 MB


,sku_numb3r,inventory_type,stocking_type,lead_time,unit_price,Manufacturing Site,division Code,transaction_date,order_quantity
581495,047A0938,FG,MTS,28,996.556745,CA-8A,DCA7,2025-04-17,81
124070,74F41574,FG,MTS,28,1018.271217,NaN,NaN,2023-06-29,67
595756,780BDC03,FG,MTS,28,982.338972,JP-F3,DB2F,2025-05-07,104
619263,2B4406EB,FG,MTS,14,50.091032,CA-8A,E477,2025-06-10,10
493699,61DB0AE2,FG,MTS,28,1003.086677,CA-8A,69A2,2024-12-11,126


- Dataset structure  
The dataset has 760,460 observations, and 9 columns:  
**sku_numb3r** - stock keeing unit number  
**inventory_type** - types of inventory (FG: finished goods, WIP: work in progress, RM: raw material)  
**stocking_type** - stocking type policy (MTS: make to stock, MTO: make to order, ATO: assemble to order)  
**lead_time** - total lead time in days  
**unit_price** - unit price in US dollars  
**Manufacturing Site** - Manufacturing site ID starting with country code from USA state  
**division Code** - four character business division ID  
**transaction_date** - date of Transaction in Year-Month-Day format  
**order_quantity** - order quantity rounded up  

Based on the sample, we can see that the column names do not follow the same naming convention. This needs to be aligned with the metadata to allow easier use in the future.  
sku_numb3r - has a 3 instead of an e --> change to sku_number  
Manufacturing Site - has capitalization and a space --> change to manufacturing_site  
division Code - has capitalization and a space --> change to division_code

In [3]:
transactions.columns = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price", "manufacturing_site", "division_code", "transaction_date", "order_quantity"]
transactions.columns

Index(['sku_number', 'inventory_type', 'stocking_type', 'lead_time',
       'unit_price', 'manufacturing_site', 'division_code', 'transaction_date',
       'order_quantity'],
      dtype='str')

**transactions.columns** refers to all the columns in the transations data frame. using **=** we replace the existing columns with the column names in the new list, to match the metadata. we need to rename the columns in the same order that they are in currently.  
Then we inspect the columns to make sure the changes were applied. 

## Question 2  Data inspection

We use the **.describe()** function to inspect the numeric oddities in the dataset. 

In [4]:
transactions.describe()

,lead_time,unit_price,order_quantity
count,760460.000000,760460.000000,760460.000000
mean,25.775126,798.052454,79.865492
std,10.724936,374.638970,41.933496
min,14.000000,-1018.428051,-135.000000
25%,28.000000,981.057439,63.000000
50%,28.000000,995.994200,93.000000
75%,28.000000,1004.126123,109.000000
max,2800.000000,1024.337563,192.000000


Looking at the table, there are three variables to inspect - lead_time, unit_price and order_quantity.  
The main thing we are looking for is whether the minimum, maximum, and quartiles make sense compared with the rest of the values.  

- In the **lead_time** column, most of the values are tightly grouped, except for the maximum value of 2800, which is extremely far from the rest of the data, considering that 75% of the observations are exactly 28. This suggests that some of the values may have accidentally added zeroes.  
We will correct this by filtering the data to show lead times over 100 days. The value of 100 was chosen as a quick way to isolate values that are clearly far above the normal range.  

- In the **unit_price** column, the minimum value is a negative number. Unit price would normally represent the price of one unit, and should normally not be below zero. The rest of the distribution is between 981 and 1024, so the value seems to be similar to the normal prices, but the sign is negative.  
We will filter the data to show values below 0, and if these appear to be normal prices with an incorrect negative sign, they can be converted to positive values.  

- In the **order_quantity** column we have another negative value in the minimum row. Though suspicious, negative quantities could potentially represent returns, cancellations, corrections... We need to inspect these first, and then determine further action. 

- **Cleaning lead_time column**

In [5]:
transactions[transactions["lead_time"] > 100]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
58529,1141EA6F,FG,MTO,2800,985.744535,JP-F3,5A6B,2023-03-26,109
134978,D9BB7F45,WIP,MTS,2800,990.760849,JP-5C,EAEE,2023-07-15,134
363558,347857BA,RM,MTO,2800,1018.551873,CA-5A,FDCB,2024-06-08,98
479436,7809F5D1,FG,MTO,2100,202.728621,CA-5A,ED3D,2024-11-21,64
509017,3789F452,WIP,MTO,2100,187.510708,JP-B0,69A2,2025-01-03,31
521854,C318378C,FG,MTO,2800,1013.996493,PL-89,DB2F,2025-01-21,81
637832,3BEBF5F1,FG,MTO,2800,992.002568,JP-B1,5DF7,2025-07-07,116
638980,B0AB8C07,RM,MTS,2800,996.056173,CA-03,DB2F,2025-07-08,114
648717,B80052D1,WIP,MTS,2800,1001.608862,CA-5A,DFD7,2025-07-23,93
732265,257728D3,FG,MTO,2800,979.357293,NaN,390D,2025-11-20,92


The datatable shows 10 values that are far above the rest of the observations. We can safely assume that two zeroes were accidentally added to these values, and that the best course for cleaning this data would be to change these values as follows:  
2100 --> 21  
2800 --> 28

In [6]:
transactions["lead_time"] = transactions["lead_time"].replace({2100:21, 2800:28})
transactions.describe()

,lead_time,unit_price,order_quantity
count,760460.000000,760460.000000,760460.000000
mean,25.740497,798.052454,79.865492
std,4.768944,374.638970,41.933496
min,14.000000,-1018.428051,-135.000000
25%,28.000000,981.057439,63.000000
50%,28.000000,995.994200,93.000000
75%,28.000000,1004.126123,109.000000
max,28.000000,1024.337563,192.000000


From the dataset transactions, we access the "lead_time" column, and using the .replace() function, we change the values as defined above.  
Using the describe function, we verify that this change was successful, and we can see that the max lead time is now 28.  

- **Cleaning unit_price column**

In [7]:
transactions[transactions["unit_price"]<0]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
49984,2AE360E8,FG,MTO,28,-987.145612,CA-5A,EAEE,2023-03-14,116
56713,4E692D00,FG,MTO,28,-994.612343,JP-5C,69A2,2023-03-23,85
264658,3247A1B9,FG,MTO,28,-982.580274,CA-5A,8E41,2024-01-18,103
315652,5BD75D98,RM,MTS,14,-47.368801,JP-FB,69A2,2024-03-31,15
431105,5E02EF37,FG,MTO,28,-1018.428051,NaN,69A2,2024-09-12,108
434302,FA4AE7D0,FG,MTO,28,-993.553563,JP-5C,DCA7,2024-09-17,98
521860,1141EA6F,FG,MTO,28,-985.744535,CA-75,5DF7,2025-01-21,133
538857,9E119356,FG,MTO,28,-990.238172,NaN,DFD7,2025-02-14,106
645799,70F56985,WIP,MTS,28,-988.974884,CA-5A,5DF7,2025-07-18,134
715103,299299C2,RM,MTS,28,-1007.214144,CA-5A,5DF7,2025-10-26,106


The table shows another 10 values with incorrect formats. All of these values seem to be normal prices, with incorrect negative values. Instead of using the .replace() function like we previously did, the .abs() function will be used to concert these to positive values. If we had used the .replace() function, we would have had to type the 10 values and their replacements manually. 

In [8]:
transactions["unit_price"] = transactions["unit_price"].abs()
transactions.describe()

,lead_time,unit_price,order_quantity
count,760460.000000,760460.000000,760460.000000
mean,25.740497,798.076113,79.865492
std,4.768944,374.588567,41.933496
min,14.000000,28.824318,-135.000000
25%,28.000000,981.057439,63.000000
50%,28.000000,995.994200,93.000000
75%,28.000000,1004.126123,109.000000
max,28.000000,1024.337563,192.000000


Checking with the describe function, we can see that the replacement was successful. 

- **Inspecting order_quantity column**  

First, we inspect the suspicious negative values in order to determine how to proceed. 

In [9]:
transactions[transactions["order_quantity"] <0]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
7,2A29C5AC,RM,MTO,14,52.286677,JP-5C,8EBD,2023-01-01,-1
36,312E94A9,FG,MTO,14,54.699918,PL-E2,DCA7,2023-01-01,-11
40,A2193035,RM,MTS,14,47.717307,JP-5C,DCA7,2023-01-01,-9
71,8FF8A9B7,WIP,MTS,21,196.854536,JP-5C,5DF7,2023-01-01,-1
78,F8C4853C,WIP,MTS,21,207.425464,CA-5A,9BA9,2023-01-01,-16
...,...,...,...,...,...,...,...,...,...
760369,8D21AF3D,RM,MTS,21,197.485532,CA-75,CCE6,2025-12-30,-7
760398,BDB5C01B,FG,MTS,14,41.303224,JP-FB,DFD7,2025-12-30,-12
760404,35BD4EA5,WIP,MTS,14,55.694193,CA-5A,8EBD,2025-12-30,-2
760436,1D0E840A,FG,MTO,14,51.831299,CA-5A,FDAC,2025-12-30,-11


Since there are many entries with negative order quantity (39,107 rows), and there is no available information to establish they are data entry errors, these values will be retained. 

## Question 3 - Missing values

In order to determine how missing values need to be delat with, we need to use .isna().sum(), so that we get the total number of NaN values in each column. 

In [10]:
transactions.isna().sum()

sku_number              2623
inventory_type         13042
stocking_type          15173
lead_time                  0
unit_price                 0
manufacturing_site    147478
division_code          53231
transaction_date           0
order_quantity             0
dtype: int64

The columns with missing values are sku_number (2,623), inventory_type (13,042), stocking_type (15,173), manufacturing_site (147,478), and division_code (53,231). The columns lead_time, unit_price, transaction_date, and order_quantity had no missing values.

The missing values in the SKU and inventory classification columns need attention because these are important for identifying products and filtering the transactions for the subsequent safety stock analysis.  

A row without **sku** cannot be assigned to an inventory item, and the eventual purpose of this assignment is safety stock analysis by SKU, so these should be dropped using .dropna(subset = ["sku_number"]).  

Dropping rows with **missing inventory classification** is reasonable too because we have no way of knowing the missing inventory type, so we cannot randomly fill them with other values. We could fill them with "Unknown", but they would be removed later anyway when we filter specifically for FG in question 5.  The same logic applies to **missing stocking type**. We remove these three because we cannot answer questions such as: What product is this? Is it finished goods? Is it make to stock?

Missing manufacturing site and division information may also limit geographic and division-level comparisons. I will avoid assigning random product identifiers or inventory classifications because doing so could make the analysis inaccurate.

In [11]:
transactions_clean = transactions.dropna(subset=["sku_number"])
transactions_clean = transactions_clean.dropna(subset=["inventory_type"])
transactions_clean = transactions_clean.dropna(subset=["stocking_type"])

We used **transactions.dropna(subset=["sku_number"])** to drop the NA values from the column sku_number. Then, subset= is used because by default it would have dropped all of the NA values from the data. A new object is defined (transactions_cleaned) to store the cleaned data, and preserve the original data in transactions, in case it needs to be referenced in the future. The same code applies to inventory types and and stocking type. 

Missing **manufacturing site and division code**  
Almost 19.4% of the dataset has missing manufacturing site, so dropping these would remove a big portion of the dataset. These rows should be preserved so we should fill this NaN information with “Unknown”, and the same procedure should be followed for the division code, since these rows need to be perserved for the safety stock analysis.

In [12]:
transactions_clean["manufacturing_site"] = transactions_clean["manufacturing_site"].fillna("Unknown")
transactions_clean["division_code"] = transactions_clean["division_code"].fillna("Unknown")

In [13]:
transactions_clean.isna().sum()

sku_number            0
inventory_type        0
stocking_type         0
lead_time             0
unit_price            0
manufacturing_site    0
division_code         0
transaction_date      0
order_quantity        0
dtype: int64

Now this is the new sum of NA values for the dataset.

## Question 4 - Extracting country values  

In order to understand how country vakues are stored, we need to examine the manufacturing_site column and it's values.

In [14]:
transactions_clean["manufacturing_site"].value_counts()

manufacturing_site
CA-5A      142386
JP-5C      142028
Unknown    141526
CA-8A      141317
JP-60       28536
US-99       28479
PL-89       13996
JP-FB        7212
US-04        7136
CA-75        7118
US-5A        7115
PL-E2        7097
JP-B0        7095
JP-B1        7058
CA-0B        7031
US-61        7017
PL-B0        6992
CA-03        6969
PL-55        6950
JP-F3        6833
Name: count, dtype: int64

From the result, we can see that the code for the manufacturing site is consisted of country codes from USA(US), Canada(CA), Poland(PL) and Japan (JP), and manufacturing site ID's (eg. 5A, 99...). First, we need to create  new column to store the extracted country codes, and assign "Unknown" to the whole column to start with.  

In [15]:
transactions_clean["country_code"] = "Unknown"

Now that we have a new column, we can extract the country codes from manufacturing_site to country_code.

In [16]:
transactions_clean.loc[transactions_clean["manufacturing_site"].str.startswith("US"), "country_code"] = "US"
transactions_clean.loc[transactions_clean["manufacturing_site"].str.startswith("CA"), "country_code"] = "CA"
transactions_clean.loc[transactions_clean["manufacturing_site"].str.startswith("PL"), "country_code"] = "PL"
transactions_clean.loc[transactions_clean["manufacturing_site"].str.startswith("JP"), "country_code"] = "JP"

**transactions_clean["manufacturing_site"]** selects the manufacturing site column from the dataset.  
**.str.startswith("US")** checks wether each value in the column starts with the defined letters. This returns boolean values for every row.   
**transactions_clean.loc[]** is used to select particular rows and culumns in a DF. In this code, rows is *transactions_clean["manufacturing_site"].str.startswith("US")* and the column is only the *country_code* column. Therefore, the whole code asks to find all rows where the manufacturing site strats with US and select their country values. Then **= "US"** assigns US to the selected cells. 

In [17]:
transactions_clean["country_code"].value_counts()

country_code
CA         304821
JP         198762
Unknown    141526
US          49747
PL          35035
Name: count, dtype: int64

We count the values in the new column to check that the code was successfully applied. There are still unknown values in the column, but that is due to the unknown values in the manufacturing site column. 

## Question 5 - Helpful Dimensional Information  

Since we ultimately want to calculate safety stock, we need to focus on finished goods designated as "make to stock." Create a filtered subset of your transaction dataset and write appropriate code to answer the following questions (for the filtered data).  

In the dataset **FG = Finished goods and MG = Make to stock**. Finished goods is in the inventory type column, and Make to stock is in the stocking type column. 

In [18]:
filtered_data = transactions_clean[(transactions_clean["inventory_type"] == "FG") & (transactions_clean["stocking_type"] == "MTS")]  
filtered_data.head()

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country_code
2,036A78DC,FG,MTS,28,999.652063,Unknown,8E41,2023-01-01,115,Unknown
3,E90CE574,FG,MTS,28,1010.156998,Unknown,E358,2023-01-01,119,Unknown
6,708D6AF0,FG,MTS,28,1003.935380,CA-8A,Unknown,2023-01-01,99,CA
9,5A83CDC4,FG,MTS,28,1005.647070,JP-B1,DFD7,2023-01-01,98,JP
10,C8E27FB5,FG,MTS,28,994.444643,US-99,9BA9,2023-01-01,98,US


Since we are looking at finished goods that are made to stock, the subset should only show rows for these two conditions. We define a new object called filtered_data where only transactions classified as FG and MTS are stored.  
**(transactions_clean["inventory_type"] == "FG")**  checks wether inventory type = finished goods  
**(transactions_clean["stocking_type"] == "MTS")** checks if stocking type = make to stock  
This creates true or false values, and the operator **&** requires both conditions to be true in order for a row to be included. 

### How many unique SKUs are we working with?  

In [19]:
filtered_data["sku_number"].value_counts()

sku_number
BDB5C01B    5395
CF33D156    3998
2249FBC0    3776
DB97FEC7    3751
8FF8A9B7    3685
            ... 
70F56985      71
1C59FAC9      71
7C7D6696      68
1141EA6F      67
1FAC49D7      67
Name: count, Length: 360, dtype: int64

From the filtered data, we select the sku number column and call the value counts function. This provides us with a Series of the sku number values and their counts in the data. The length of the series is 360, which means there are **360 unique SKUs**.

### How many unique Manufacturing Sites are we working with?

In [20]:
filtered_data["manufacturing_site"].value_counts()

manufacturing_site
CA-5A      50946
Unknown    50321
JP-5C      50247
CA-8A      50244
JP-60      10134
US-99      10092
PL-89       5024
US-04       2623
JP-FB       2553
PL-E2       2550
PL-B0       2522
JP-B1       2517
JP-B0       2513
CA-75       2492
US-5A       2488
CA-0B       2473
PL-55       2465
CA-03       2434
US-61       2410
JP-F3       2349
Name: count, dtype: int64

From the filtered datam we get a Series that lists the uique manufacturing sites. 

In [21]:
filtered_data["manufacturing_site"].value_counts().count()

np.int64(20)

To count the length of the series, we apply the .count() function on top of that and the result is **20 unique manufacturing sites**, including unknown values, so . 

### How many Divisions are we working with?

In [22]:
filtered_data["division_code"].value_counts().count()

np.int64(55)

We apply the same logic for the division codes column, and the result is **55 unique divisions**, again including unknown values.

### What are the top and bottom 10 transactions by order quantity?  

For the bottom 10, we sort the values of the order quantity column, followed by .head(10) to show the first 10 observations. The **.sort_values** function automatically sorts by ascending, so the orders with the lowest order quantity will show up on top. 

In [23]:
filtered_data.sort_values("order_quantity").head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country_code
411026,CF33D156,FG,MTS,28,1021.354856,Unknown,A866,2024-08-15,-95,Unknown
75402,7747965D,FG,MTS,28,980.002831,CA-5A,3D40,2023-04-20,-95,CA
41032,00D0DF3B,FG,MTS,28,1010.057470,CA-8A,ACEE,2023-03-01,-93,CA
238079,A093EB12,FG,MTS,28,998.281566,JP-5C,8E41,2023-12-11,-88,JP
507799,04CD4C8F,FG,MTS,21,212.250683,Unknown,69A2,2025-01-01,-63,Unknown
191297,6BCB8698,FG,MTS,21,198.135265,JP-60,9BA9,2023-10-04,-58,JP
746699,CCD3552E,FG,MTS,21,208.118486,US-99,DB2F,2025-12-11,-58,US
581208,19FB77E3,FG,MTS,21,199.756981,US-61,9BA9,2025-04-16,-54,US
753183,8760340C,FG,MTS,21,206.923468,JP-60,69A2,2025-12-20,-49,JP
625425,EF035773,FG,MTS,21,195.176706,PL-B0,Unknown,2025-06-19,-47,PL


All the values of the bottom 10 transactions by order quantity are negative, so -95 shows up on top.

To sort the data by descending, so that the biggest values show up on top, we need to add **ascending= False** to the sort values function. 

In [24]:
filtered_data.sort_values("order_quantity", ascending = False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country_code
332825,361B3A3C,FG,MTS,28,994.572406,CA-5A,DFD7,2024-04-25,189,CA
417833,EF1EBAE2,FG,MTS,28,999.393166,Unknown,5DF7,2024-08-24,187,Unknown
566007,9DD9199E,FG,MTS,28,1009.445290,CA-75,4BA1,2025-03-25,187,CA
196131,4626B062,FG,MTS,28,987.536273,JP-5C,CCE6,2023-10-11,186,JP
122621,4626B062,FG,MTS,28,987.536273,CA-8A,E358,2023-06-27,185,CA
62719,BDB5C01B,FG,MTS,28,992.623768,CA-8A,CCE6,2023-04-01,181,CA
165676,8FF8A9B7,FG,MTS,28,991.330046,CA-5A,EAEE,2023-08-28,181,CA
455991,E1E201E9,FG,MTS,28,1012.145121,CA-5A,8E41,2024-10-18,180,CA
722303,2077D3E8,FG,MTS,28,1002.666038,US-99,EAEE,2025-11-05,179,US
575090,0BC11363,FG,MTS,28,1018.203896,JP-5C,69A2,2025-04-07,179,JP


### What are the top and bottom 10 transactions by total sales value?  

Since sales is not a feature in this dataset, we need to calculate the sales by multyplying the columns unit price and order quantity. We also need to create a new column to store these values. 

In [25]:
filtered_data["total_sales_value"] = (filtered_data["order_quantity"] * filtered_data["unit_price"])
filtered_data.sample(5)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country_code,total_sales_value
527165,D622EA37,FG,MTS,28,999.573287,CA-5A,DCA7,2025-01-29,88,CA,87962.449222
506096,036A78DC,FG,MTS,28,999.652063,JP-5C,DCCD,2024-12-29,112,JP,111961.031100
718004,4626B062,FG,MTS,28,987.536273,CA-8A,Unknown,2025-10-30,108,CA,106653.917431
489402,E6CF453B,FG,MTS,28,1001.134185,Unknown,69A2,2024-12-05,61,Unknown,61069.185262
604510,EAEE18E6,FG,MTS,28,1009.171987,Unknown,DCA7,2025-05-20,128,Unknown,129174.014370


We use .sample(5) to show 5 rando rows in the dataset and check that the new column was added with the right values.  
From here, we can apply the same code as the previous question to show the top 10 and bottom 10 orders by sales value.  

Top 10:

In [26]:
filtered_data.sort_values("total_sales_value", ascending = False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country_code,total_sales_value
566007,9DD9199E,FG,MTS,28,1009.445290,CA-75,4BA1,2025-03-25,187,CA,188766.269262
332825,361B3A3C,FG,MTS,28,994.572406,CA-5A,DFD7,2024-04-25,189,CA,187974.184791
417833,EF1EBAE2,FG,MTS,28,999.393166,Unknown,5DF7,2024-08-24,187,Unknown,186886.521965
196131,4626B062,FG,MTS,28,987.536273,JP-5C,CCE6,2023-10-11,186,JP,183681.746686
122621,4626B062,FG,MTS,28,987.536273,CA-8A,E358,2023-06-27,185,CA,182694.210413
575090,0BC11363,FG,MTS,28,1018.203896,JP-5C,69A2,2025-04-07,179,JP,182258.497461
455991,E1E201E9,FG,MTS,28,1012.145121,CA-5A,8E41,2024-10-18,180,CA,182186.121808
608057,E1E201E9,FG,MTS,28,1012.145121,CA-5A,17D3,2025-05-25,178,CA,180161.831566
62719,BDB5C01B,FG,MTS,28,992.623768,CA-8A,CCE6,2023-04-01,181,CA,179664.901949
722303,2077D3E8,FG,MTS,28,1002.666038,US-99,EAEE,2025-11-05,179,US,179477.220864


Bottom 10:

In [27]:
filtered_data.sort_values("total_sales_value").head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country_code,total_sales_value
411026,CF33D156,FG,MTS,28,1021.354856,Unknown,A866,2024-08-15,-95,Unknown,-97028.711351
41032,00D0DF3B,FG,MTS,28,1010.057470,CA-8A,ACEE,2023-03-01,-93,CA,-93935.344750
75402,7747965D,FG,MTS,28,980.002831,CA-5A,3D40,2023-04-20,-95,CA,-93100.268967
238079,A093EB12,FG,MTS,28,998.281566,JP-5C,8E41,2023-12-11,-88,JP,-87848.777852
507799,04CD4C8F,FG,MTS,21,212.250683,Unknown,69A2,2025-01-01,-63,Unknown,-13371.793040
746699,CCD3552E,FG,MTS,21,208.118486,US-99,DB2F,2025-12-11,-58,US,-12070.872198
191297,6BCB8698,FG,MTS,21,198.135265,JP-60,9BA9,2023-10-04,-58,JP,-11491.845367
581208,19FB77E3,FG,MTS,21,199.756981,US-61,9BA9,2025-04-16,-54,US,-10786.876960
753183,8760340C,FG,MTS,21,206.923468,JP-60,69A2,2025-12-20,-49,JP,-10139.249950
746931,1144D57C,FG,MTS,21,214.758781,Unknown,Unknown,2025-12-11,-46,Unknown,-9878.903907


## References  

No outside references were used in this assignment